# L08-02　ATC 转换命令与参数

L08-01 用过一条最小的 ATC 命令，对必填参数进行了介绍，没说为什么这么写。本节把这条命令拆开，逐个参数讲清取值规则和写错的后果。

**前置**：已完成 L08-01，`l08_workspace/demo_model.onnx` 已生成。

**环境**：第 1~4 节只讲参数写法，任意机器可读；第 5 节需要昇腾开发套件。

## 1　命令骨架

ATC 的参数形式统一是 `--参数=值`：

```bash
atc --model=xxx.onnx --framework=5 --output=xxx --soc_version=xxx
```

参数总数上百个，不需要记住，只需要知道哪些必填、其余去哪里查。按用途分组：

| 组别 | 代表参数 | 作用 |
| --- | --- | --- |
| 输入 | `--model` `--framework` `--input_shape` `--input_format` | 输入是什么 |
| 输出 | `--output` `--output_type` | 产物放哪、什么类型 |
| 目标 | `--soc_version` | 为哪块芯片编译 |
| 精度 | `--precision_mode` `--precision_mode_v2` | 精度与性能取舍 |
| 动态 shape | `--dynamic_batch_size` `--dynamic_image_size` `--dynamic_dims` `--input_shape_range` | 支持可变输入 |
| 调试 | `--log` `--debug_dir` | 排查转换问题 |
| 调优 | `--op_select_implmode` `--buffer_optimize` `--fusion_switch_file` | 性能调优 |

参数在 CANN 版本间会变，最权威的来源是本机的 `atc --help`。

In [1]:
!source /usr/local/Ascend/ascend-toolkit/set_env.sh && atc --help | grep -c '^\s*--'

74


### 2.1　`--model`

输入模型的路径。取什么文件由 `--framework` 决定：ONNX 是 `.onnx`，TensorFlow 是 `.pb`，Caffe 的结构与权重分离，`--model` 给 `.prototxt`，另外还要用 `--weight` 给 `.caffemodel`。

两个常见问题：用相对路径但工作目录不对；路径含空格没加引号。

### 2.2　`--framework`

告诉 ATC 用哪个解析器读模型。

| 值 | 框架 | 典型文件 |
| --- | --- | --- |
| 0 | Caffe | `.prototxt` + `.caffemodel` |
| 1 | MindSpore | `.air` / MindIR |
| 3 | TensorFlow | `.pb` |
| 5 | ONNX | `.onnx` |

2 和 4 不使用，编号不连续，容易让人误以为可以顺推。

写错的后果是用错误的解析器去读文件，报解析失败，**而报错信息不会提示是 framework 填错了**。本课程只用 5。

### 2.3　`--output`

输出路径的**前缀**，不带扩展名，ATC 自动追加 `.om`。

```bash
--output=model         # 得到 model.om
--output=model.om      # 得到 model.om.om
```

目录必须已存在，ATC 不会自动创建。

实用建议：把关键参数编进文件名，比如 `resnet50_bs1_fp16`。同一个模型转多个版本时，靠文件名区分比靠记忆可靠。

### 2.4　`--input_shape`

指定输入节点的名称与形状，语法是 `"名称:维度,维度,..."`。

**多输入用分号分隔**，不是逗号：

```bash
--input_shape="input_ids:1,128;attention_mask:1,128;token_type_ids:1,128"
```

整个值必须用双引号包住。不加引号时 shell 会把分号当命令分隔符，命令在第一个分号处被截断，后面的部分被当成新命令执行——报错信息会非常莫名。

名称必须与 ONNX 里的 `input_names` 完全一致。最可靠的做法是从模型里读，而不是手抄：

In [2]:
import onnx

m = onnx.load("l08_workspace/demo_model.onnx")

parts = []
for i in m.graph.input:
    dims = [d.dim_value if d.dim_value else -1 for d in i.type.tensor_type.shape.dim]
    parts.append("%s:%s" % (i.name, ",".join(str(d) for d in dims)))

print('--input_shape="%s"' % ";".join(parts))

--input_shape="input:1,3,32,32"


输出里出现 `-1` 说明该维在 ONNX 里是动态的，这时必须配合 `--dynamic_*` 参数给出档位，见第 4 节。

### 2.5　`--soc_version`

目标芯片型号，决定编译目标。不写会报：

```text
the required parameter [--soc_version] for ATC is empty
```

它是编译期参数而不是运行期参数。回到 L08-01 的编译器类比：OM 相当于针对特定 CPU 编译出的可执行文件，型号不匹配的 OM 加载会直接失败。这就是换芯片必须重新转换的原因。

取值只能实测，`npu-smi info` 或 `acl.get_soc_name()`。

## 3　常用可选参数

### `--input_format`

取 `NCHW`、`NHWC` 或 `ND`。PyTorch 导出的 CV 模型通常是 NCHW；NLP 模型和非图像张量用 ND。MindSpore 只支持 NCHW。

它与 `--input_shape` 的维度顺序必须自洽。两者不一致时 ATC 不一定报错，但结果是错的——这类"不报错但结果错"的问题最难查。

### `--log`

取 `debug` / `info` / `warning` / `error`。平时用 `info`，转换失败时立刻改 `debug`。debug 日志是 L08-03 做错误定位的主要材料。

### `--precision_mode` 与 `--precision_mode_v2`

全局精度模式，直接影响精度和性能。

`--precision_mode` 的常见取值：`force_fp16`、`allow_fp32_to_fp16`、`must_keep_origin_dtype`、`allow_mixed_precision`、`force_fp32`。

`--precision_mode_v2` 是新版参数，取值 `fp16`、`origin`、`mixed_float16` 等。

**两者互斥，不能同时配置。** 默认值随 CANN 版本和芯片型号变化，用本机 `atc --help` 确认，不要照抄文档里的默认值。

怀疑精度问题时有个实用套路：用 `must_keep_origin_dtype` 再转一版做对照。

In [3]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_fp32 \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B4 \
    --precision_mode=must_keep_origin_dtype

ATC start working now, please wait for a moment.
.

/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_ops_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The 

path string is NULLpath string is NULL...
ATC run success, welcome to the next use.



两版 OM 都跑一遍 msame，用 L08-01 对齐 ② 的方法比较。如果 `must_keep_origin_dtype` 版本精度正常、默认版本不正常，说明是精度模式导致的；如果两版都不正常，问题在转换本身或模型本身。

代价是这一版性能会明显下降，它只用于排查，不用于部署。

## 4　动态 shape

batch 随请求合批变化、图像分辨率不固定、序列长度不定，这些场景下 shape 无法在编译期确定。

四种模式：

| 模式 | ATC 参数 | `--input_shape` 写法 | 适用 |
| --- | --- | --- | --- |
| 静态 | 无 | 全部写实际值 | 输入固定，性能最优 |
| 动态 batch | `--dynamic_batch_size="1,2,4,8"` | batch 维写 `-1` | 只有 batch 变 |
| 动态分辨率 | `--dynamic_image_size="224,224;448,448"` | H/W 维写 `-1` | 图像尺寸变 |
| 动态维度 | `--dynamic_dims="16;32;64"` | 对应维写 `-1` | 任意指定维度变 |
| 完全动态 | `--input_shape_range="input:[-1,3,-1,-1]"` | 用 range 语法 | shape 完全不定 |

四条规则：

1. `--dynamic_*` 必须与 `--input_shape` 配合，可变维在 `--input_shape` 里写 `-1`
2. 三个 `--dynamic_*` 参数互相排斥，只能选一个
3. `--dynamic_dims` 还需与 `--input_format` 配合
4. 档位之间用分号分隔，档位内部用逗号

In [5]:
import torch
import torch.nn as nn

# 创建工作目录
import os
os.makedirs("l08_workspace", exist_ok=True)


# 一个简单 CNN 模型
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 3, kernel_size=3, padding=1),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        return self.net(x)


model = DemoModel()
model.eval()


# dummy input
dummy_input = torch.randn(1, 3, 32, 32)


# 导出动态 batch ONNX
torch.onnx.export(
    model,
    dummy_input,
    "l08_workspace/demo_model_dyn.onnx",
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {
            0: "batch"
        },
        "output": {
            0: "batch"
        }
    }
)

print("ONNX export success")

/tmp/ipykernel_4865/3103132896.py:33: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0814 15:58:42.317000 4865 site-packages/torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DemoModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DemoModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/opt/atomgit/.local/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/atomgit/.local/lib/python3.11/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/opt/atomgit/.local/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/atomgit/.local/lib/python3.11/site

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX export success


In [6]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model_dyn.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_dyn \
    --input_format=NCHW \
    --input_shape="input:-1,3,32,32" \
    --dynamic_batch_size="1,2,4,8" \
    --soc_version=Ascend910B4 

ATC start working now, please wait for a moment.
...

/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_ops_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The 

path string is NULLpath string is NULL...
ATC run success, welcome to the next use.



档位不是越多越好。每个档位都要单独编译一份执行序列，OM 体积和转换时间随档位数量增长。按实际业务的请求分布选 3~5 档，覆盖主要场景即可。

## 5　转换前的自查

ATC 转换耗时从几分钟到几十分钟，因为一个拼写错误白等一轮不值得。提交前对照下面这几条自查，比事后看报错快得多。

| 检查项 | 常见错误 |
| --- | --- |
| 必填参数齐全 | 漏 `--soc_version` |
| `--output` 不带 `.om` | 写成 `model.om`，得到 `model.om.om` |
| 模型后缀与 framework 匹配 | `.onnx` 配 `--framework=3` |
| `--input_shape` 语法 | 多输入用了逗号；整个值没加引号 |
| 输入名与 ONNX 一致 | 手抄错字；模型重新导出后名字变了 |
| `-1` 与 `--dynamic_*` 配套 | 有 `-1` 却没给档位；给了档位却没写 `-1` |
| `--dynamic_*` 互斥 | 同时配了 batch 和 image_size |
| 精度模式互斥 | `--precision_mode` 与 `_v2` 同时出现 |
| `soc_version` 是实测值 | 照抄了文档里的 `Ascend310` |
| 输出目录已存在 | ATC 不会自动创建 |

前两项和输入名这项占了错误的大多数。

In [7]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1
atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model_bs1_fp16 \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B4\
    --log=info

echo "退出码: $?"
ls -lh l08_workspace/*.om

ATC start working now, please wait for a moment.
..

/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_ops_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The 

path string is NULLpath string is NULL...
ATC run success, welcome to the next use.

退出码: 0
-rw------- 1 atomgit atomgitgroup  90K Aug 14 15:42 l08_workspace/demo_model.om
-rw------- 1 atomgit atomgitgroup  90K Aug 14 16:00 l08_workspace/demo_model_bs1_fp16.om
-rw------- 1 atomgit atomgitgroup 200K Aug 14 15:59 l08_workspace/demo_model_dyn.om
-rw------- 1 atomgit atomgitgroup  85K Aug 14 15:54 l08_workspace/demo_model_fp32.om


## 6　三个完整案例

**CV 分类，静态 shape**

```bash
atc --model=resnet50.onnx --framework=5 \
    --output=resnet50_bs1 \
    --input_format=NCHW --input_shape="input:1,3,224,224" \
    --soc_version=<实测值>
```

**CV 检测，动态分辨率**

```bash
atc --model=yolov5.onnx --framework=5 \
    --output=yolov5_dynres \
    --input_format=NCHW --input_shape="images:1,3,-1,-1" \
    --dynamic_image_size="640,640;1280,1280" \
    --soc_version=<实测值>
```

**NLP 多输入，动态 batch**

```bash
atc --model=bert.onnx --framework=5 \
    --output=bert_dynbs \
    --input_format=ND \
    --input_shape="input_ids:-1,128;attention_mask:-1,128;token_type_ids:-1,128" \
    --dynamic_batch_size="1,4,8,16" \
    --soc_version=<实测值>
```

三个案例覆盖了大部分实际场景的参数组合。注意第三个：多输入用分号分隔、非图像张量用 ND、每个输入的 batch 维都要写 `-1`。

## 7　小结

| 要回答的问题 | 对应参数 |
| --- | --- |
| 模型是什么 | `--model` `--framework` `--input_shape` `--input_format` |
| 跑在哪 | `--soc_version` |
| 怎么取舍 | `--precision_mode` `--dynamic_*` |
| 产物放哪 | `--output` |



参数速查：

```text
必填  --model --framework --output --input_shape --soc_version
常用  --input_format --log --precision_mode
动态  --dynamic_batch_size | --dynamic_image_size | --dynamic_dims   (三者互斥)
      --input_shape_range                                            (完全动态)
调试  --log=debug --debug_dir
```

## 8　练习

1. 为一个三输入的 NLP 模型写出正确的 `--input_shape`，输入名分别是 `input_ids`、`attention_mask`、`token_type_ids`，形状都是 `(1, 128)`。
2. 找出下面命令的三处错误：
   ```bash
   atc --model=model.onnx --framework=3 --output=model.om \
       --input_shape="a:1,3,224,224,b:1,10" --soc_version=Ascend310
   ```
3. 把 L08-01 的静态命令改成支持 1/2/4/8 四个 batch 档位，注意 ONNX 也要重新导出。
4. 用 `must_keep_origin_dtype` 和默认精度各转一版，跑 msame 比较输出差异，记录实际数值。
5. 在昇腾环境跑 `atc --help`，核对第 7 节速查表里的参数是否都存在，记录差异。
6. 为什么档位越多 OM 越大、转换越慢？结合 OM 的组成回答。